# 91장 · 키워드 조회 함수의 최소 구현

최소 조회 함수는 일부 질문에서만 맞을 수 있습니다. 장비명이 같은 두 질문을 보내 목적까지 구별하는지 확인합니다.

Q01이 입력에서 규정 한 줄과 출처를 반환하기까지 최소 경로를 구현합니다. 같은 장비명으로 교체 일정을 물었을 때도 점검 규정을 내놓는 반례로 범위를 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## thin_lookup()의 장비명·그룹 조건

91장의 첫 구현은 장비명과 허용 그룹으로 규정을 찾습니다. 먼저 질문을 넣었을 때 화면에 어떤 답이 돌아오는지 보고, 서버와 모델 호출은 뒤로 미룹니다. 아래 셀은 공개 그룹의 장비 질문 경로를 재현합니다. 응답의 sources에는 아직 문서 ID만 담습니다.


### 셀 01 · thin_lookup()의 정상 응답

A가 들어간 질문을 넣고 어떤 필드가 화면으로 넘어가는지 읽어 보세요.


In [1]:
documents = [{'id': 'RULE-A',
  'equipment': '장비 A',
  'text': '점검 전 전원 표시등을 확인합니다.',
  'source': '가상 장비실 규정 A',
  'page': 1,
  'section': '2.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-B',
  'equipment': '장비 B',
  'text': '점검 전 고정 장치를 확인합니다.',
  'source': '가상 장비실 규정 B',
  'page': 2,
  'section': '2.2',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-ADMIN',
  'equipment': None,
  'text': '관리자 전용 절차는 선임 검토 기록을 확인하는 것입니다.',
  'source': '가상 장비실 내부 규정',
  'page': 3,
  'section': '3.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['admin']}]

def thin_lookup(question):
    if not isinstance(question, str) or not question.strip():
        return {"status": "invalid_input", "answer": "질문을 입력하세요.", "sources": []}
    matches = [d for d in documents if d["equipment"] and
               d["equipment"] in question and "public" in d["groups"]]
    if len(matches) != 1:
        return {"status": "need_clarification", "answer": "장비 A 또는 장비 B 중 하나를 지정하세요.", "sources": []}
    doc = matches[0]
    return {"status": "answered", "answer": doc["text"], "sources": [{"id": doc["id"]}]}

reply = thin_lookup("장비 A의 점검 전 확인 사항은?")
print(reply["answer"], reply["sources"])

점검 전 전원 표시등을 확인합니다. [{'id': 'RULE-A'}]


전원 표시등과 RULE-A가 찍힙니다. 아직 화면에서 2.1절이나 개정일을 찾을 수 없습니다. 88장에서 합의한 출처 기준을 충족하려면 응답에 그 값도 담아야 합니다.


#### thin_lookup()이 장비 A의 규정 하나를 고르는 과정

```mermaid
flowchart TD
ninput["입력 검사"]
  nfilter["공개 장비 검색"]
  ncount["후보 수"]
  nask["추가 질문"]
  nreply["규정 반환"]
  ninput --> nfilter
  nfilter --> ncount
  ncount --> nask
  ncount --> nreply
```

공개 문서에서 장비 이름이 일치하는 항목을 셉니다. 일치한 문서가 정확히 한 건일 때만 그 문장을 답에 담습니다.

## 검색 의도 불일치와 오답

장비명은 맞지만 질문의 목적이 다른 경우입니다. 규정 A의 문장이 사실이어도 교체 일정의 근거로는 쓸 수 없습니다.

<!-- lesson-visual practice-space -->
**장비명이 맞아도 답이 엉뚱해질 수 있을까요?**

thin_lookup()이 검사하는 것은 공개 문서의 장비명입니다. 같은 장비에 대해 점검과 교체 일정을 각각 물어보세요.

장비 A의 점검과 교체 일정 모두 RULE-A로 연결됩니다. 두 번째 질문은 찾은 문서가 있어도 질문의 의도와 답변이 어긋납니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter91/#practice-space)
<!-- /lesson-visual practice-space -->


### 셀 02 · 의도 불일치 반례

함수는 그대로 두고 질문만 바꿉니다.

인계 기록을 둘로 나눠 적으세요. Q01은 답 문장은 맞지만 출처의 절이 빠졌습니다. Q03은 교체 일정을 물었는데도 표시등 안내가 나왔습니다. 다음 팀원이 고칠 곳은 각각 출처 위치와 질문 의도입니다.


In [2]:
question = '장비 A를 다음 달에 교체하기로 확정했나요?'
print(thin_lookup(question)["answer"])

점검 전 전원 표시등을 확인합니다.


점검 문장이 또 나옵니다. 함수는 장비 이름을 발견한 뒤 ‘점검’과 ‘교체’ 중 무엇을 묻는지 보지 않았습니다. 이 응답은 다음 장의 실패 사례입니다.

질문의 의도를 바꿔도 ‘전원 표시등’이라는 점검 안내가 그대로 나옵니다. 정상 예제 한 건이 성공한 것으로는 이 함수가 사용자의 의도를 구분한다고 말할 수 없습니다. 최소 시연을 만든 다음에는 장비는 같고 요청만 다른 입력을 붙여 처리 범위를 드러냅니다.


#### 규칙에 맞는 문장을 찾고도 오답이 되는 경우

<mark>장비명이 맞는 것과 질문의 의도가 맞는 것은 별개의 조건입니다.</mark> 같은 장비에 대해 점검 방법과 교체 일정을 각각 물어 보세요. 두 질문에 같은 규정 문장을 반환했다면 검색 성공으로 셀 질문은 어느 쪽인지 정합니다.

선수 개념 · 45장 질문 전체 일치 검색의 기준선과 빈 결과

### 과제 03 · 선택 연습 · 장비명 누락 처리

어느 장비인지 묻는 경로로 들어가려면 어떤 입력을 주면 될까요? 상태값까지 확인하세요.


In [3]:
# thin_lookup에 장비명 없는 점검 질문을 넣어 보세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(thin_lookup(&quot;점검 전 확인 사항은?&quot;)[&quot;status&quot;])</code></pre>
<p>실행 결과</p><pre><code>need_clarification</code></pre>
<p>장비가 하나로 정해지지 않아 need_clarification이 나옵니다. 이때 답변을 추측하지 않고 장비 이름을 요청할 수 있습니다.</p>
</details>


## 검색 함수를 요청 한 건을 받는 API에 붙이기

thin_lookup에서 Q01이 동작했습니다. 이번에는 같은 함수가 반환한 답이 HTTP 응답의 어느 칸에 오는지 확인합니다. 서버를 외부에 띄우지 않고 노트북 안에서 FastAPI의 테스트 클라이언트를 사용합니다.

### 셀 05 · 테스트 클라이언트로 Q01 요청

수업 환경에 FastAPI와 호환되는 httpx가 필요합니다. 아래의 question은 요청 본문, reply는 서버가 보낸 JSON입니다.

In [4]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

class InspectionQuery(BaseModel):
    question: str

demo_api = FastAPI()
@demo_api.post("/inspection")
def inspect_rule(query: InspectionQuery):
    return thin_lookup(query.question)

with TestClient(demo_api) as client:
    http_reply = client.post("/inspection", json={"question": "장비 A의 점검 전 확인 사항은?"})
body = http_reply.json()
print(http_reply.status_code, body["answer"], body["sources"][0]["id"])

200 점검 전 전원 표시등을 확인합니다. RULE-A


200 옆의 점검 문장과 RULE-A는 HTTP 클라이언트가 실제 응답 본문에서 꺼낸 값입니다. 다만 이 기준선은 출처의 절·개정일과 질문 목적을 아직 검사하지 않습니다.

### 팀별 경계 정하기

팀의 기존 조회 함수를 다른 함수로 감싸고 테스트 요청을 보내세요. 질문 필드를 통째로 뺀 요청과 빈 문자열을 넣은 요청을 비교합니다. 처음 것은 모델 검증에서, 두 번째 것은 thin_lookup 안에서 처리됩니다.

<mark>HTTP 요청 성공과 업무 기준 통과를 다른 칸에 기록합니다.</mark> 같은 경로로 Q03을 보내도 이 기준선은 200을 반환하면서 잘못된 점검 안내를 줄 수 있습니다.

92장은 HTTP 코드와 별도로 평가표에서 어떤 문항의 상태·출처가 맞았는지 셉니다.

## 돌아보기

Q03의 잘못된 답과 Q01의 빠진 출처 위치를 그대로 저장하세요. 92장에서 동일한 입력으로 어디까지 동작했는지 세어 봅니다.
